In [29]:
import io, sys, os, shutil
import pandas as pd

# 1. Setup FleetPy root on sys.path
NOTEBOOK_DIR = os.getcwd()
FLEETPY_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "../../"))
sys.path.append(FLEETPY_ROOT)

# 2. Imports
from src.misc.config import ConstantConfig, ScenarioConfig
from src.misc.init_modules import load_simulation_environment
from src.evaluation.tutorial_analysis import analyze_kpis

# 3. Copy the existing example demand into a temporary in-memory location
#    so that FleetPy's demand loader can find it under data/demand/example_demand/matched/example_network/
TEMP_DEMAND_DIR = os.path.join(FLEETPY_ROOT, "data", "demand", "example_demand", "matched", "example_network")
os.makedirs(TEMP_DEMAND_DIR, exist_ok=True)
shutil.copy(
    os.path.join(FLEETPY_ROOT, "data", "demand", "example_demand", "example_demand.csv"),
    os.path.join(TEMP_DEMAND_DIR, "example_demand.csv")
)

# 4. Build constant config and scenario config CSVs in memory
constant_csv = io.StringIO()
constant_df = pd.DataFrame([
    ("sim_env", "BatchOfferSimulation"),
    ("network_name", "example_network"),
    ("demand_name", "example_demand"),
    ("start_time", 0),
    ("end_time", 3600),
    ("time_step", 60),
    ("user_max_decision_time", 60),
    ("op_fleet_composition", "default_vehtype:10"),
    ("op_module", "RidePoolingBatchAssignmentFleetcontrol"),
    ("op_rp_batch_optimizer", "AlonsoMora"),
    ("op_vr_control_func_dict", "{'func_key':'distance_and_user_times_with_walk','vot':0.45}"),
    ("rq_type", "BasicRequest"),
    ("network_type", "NetworkBasic"),
    ("op_max_wait_time", 300),
    ("op_max_detour_time_factor", 1.4),
    ("op_const_boarding_time", 30),
    ("nr_mod_operators", 1),
    ("n_cpu_per_sim", 1),
    ("route_output_flag", False),
    ("replay_flag", False),
    ("log_level", "info"),
    ("random_seed", 1),
], columns=["Parameter","Value"])
constant_df.to_csv(constant_csv, index=False)
constant_csv.seek(0)

scenario_csv = io.StringIO()
scenario_df = pd.DataFrame([{
    "study_name": "uc2_age_based_test",
    "scenario_name": "batch_example_1",
    "rq_file": "example_demand.csv",
    "op_fleet_composition": "default_vehtype:10",
    "op_rp_batch_optimizer": "AlonsoMora"
}])
scenario_df.to_csv(scenario_csv, index=False)
scenario_csv.seek(0)

# 5. Load configs from the in-memory CSVs
constant_cfg = ConstantConfig(constant_csv)
scenario_cfgs = ScenarioConfig(scenario_csv)
scenario_cfg = constant_cfg + scenario_cfgs[0]

# 6. Run the simulation
sim = load_simulation_environment(scenario_cfg)
sim.run()

# 7. KPI analysis
results_dir = os.path.join(FLEETPY_ROOT, "results", scenario_cfg["scenario_name"])
kpis = analyze_kpis(results_dir)
print("📈 KPI Summary:")
print(kpis.to_string(index=False))


OSError: [Errno 30] Read-only file system: '/data'